# 09 — Model & Data Quality Assessment

**Purpose:** Sense-check all ML models and key data metrics to ensure predictions and outputs are sound.  

Models assessed:
- K-means customer segmentation (cluster quality, balance, spend ordering)
- Logistic regression churn classifier (accuracy, precision, recall, F1, AUC)
- Linear regression CLV predictor (R-squared, realistic predictions)

Data quality:
- Market share consistency per category
- Retention bounds
- Cross-model alignment (segments vs churn vs CLV vs momentum)

In [ ]:
from google.cloud import bigquery
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

import os
PROJECT = os.environ.get('BQ_PROJECT', 'fmn-sandbox')
LOCATION = 'africa-south1'
client = bigquery.Client(project=PROJECT, location=LOCATION)

def q(sql):
    return client.query(sql).to_dataframe()

passed = 0
failed = 0
warnings = 0

def check(name, condition, detail=''):
    global passed, failed
    if condition:
        print(f'  PASS: {name}')
        passed += 1
    else:
        print(f'  FAIL: {name} {detail}')
        failed += 1

def warn(name, detail=''):
    global warnings
    print(f'  WARN: {name} {detail}')
    warnings += 1

print(f'Connected to {PROJECT} ({LOCATION})')

## 1. K-Means Segmentation Validation

In [ ]:
print('--- K-MEANS SEGMENTATION ---\n')

# model quality
try:
    km_eval = q(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT}.analytics.kmeans_customer_segments`)')
    db_index = km_eval.iloc[0]['davies_bouldin_index']
    print(f'  Davies-Bouldin Index: {db_index:.4f}')
    check('DB index < 2.0 (good clustering)', db_index < 2.0, f'got {db_index:.4f}')
    check('DB index < 1.5 (great clustering)', db_index < 1.5, f'got {db_index:.4f}')
except Exception as e:
    print(f'  Could not evaluate model: {e}')

# segment balance
seg_dist = q(f"""
    SELECT segment_name, COUNT(*) AS n,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM `{PROJECT}.marts.mart_cluster_output`
    GROUP BY segment_name ORDER BY n DESC
""")

display(seg_dist)
max_pct = seg_dist['pct'].max()
min_pct = seg_dist['pct'].min()
check('No segment > 40% of total (balanced)', max_pct < 40, f'largest is {max_pct}%')
check('No segment < 5% of total (not too small)', min_pct > 5, f'smallest is {min_pct}%')
check('Exactly 5 segments', len(seg_dist) == 5, f'got {len(seg_dist)}')

# spending order makes sense
spend_order = q(f"""
    SELECT segment_name, ROUND(AVG(val_trns), 0) AS avg_spend
    FROM `{PROJECT}.marts.mart_cluster_output`
    GROUP BY segment_name ORDER BY avg_spend DESC
""")
display(spend_order)
check('Champions have highest avg spend', spend_order.iloc[0]['segment_name'] == 'Champions')
check('Dormant have lowest avg spend', spend_order.iloc[-1]['segment_name'] == 'Dormant')

## 2. Churn Model Validation

In [ ]:
print('--- CHURN CLASSIFIER ---\n')

try:
    ch_eval = q(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT}.analytics.churn_classifier`)')
    e = ch_eval.iloc[0]
    acc = float(e.get('accuracy', 0))
    f1 = float(e.get('f1_score', 0))
    prec = float(e.get('precision', 0))
    rec = float(e.get('recall', 0))
    auc = float(e.get('roc_auc', 0))

    print(f'  Accuracy:  {acc:.4f}')
    print(f'  F1 Score:  {f1:.4f}')
    print(f'  Precision: {prec:.4f}')
    print(f'  Recall:    {rec:.4f}')
    print(f'  ROC AUC:   {auc:.4f}')
    print()

    check('Accuracy > 0.70', acc > 0.70, f'got {acc:.4f}')
    check('Accuracy < 0.98 (not overfit)', acc < 0.98, f'got {acc:.4f} - suspiciously high')
    check('F1 > 0.40 (balanced performance)', f1 > 0.40, f'got {f1:.4f}')
    check('ROC AUC > 0.70', auc > 0.70, f'got {auc:.4f}')
    check('Precision > 0.30 (not too many false positives)', prec > 0.30, f'got {prec:.4f}')
    check('Recall > 0.30 (catching enough churners)', rec > 0.30, f'got {rec:.4f}')

except Exception as e:
    print(f'  Could not evaluate: {e}')

# risk distribution sanity
risk_dist = q(f"""
    SELECT churn_risk_level, COUNT(*) AS n,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS pct
    FROM `{PROJECT}.marts.mart_churn_risk`
    GROUP BY churn_risk_level
""")
display(risk_dist)

crit_pct = float(risk_dist[risk_dist['churn_risk_level'] == 'Critical']['pct'].values[0]) if 'Critical' in risk_dist['churn_risk_level'].values else 0
check('Critical risk < 20% of total (not overreacting)', crit_pct < 20, f'got {crit_pct}%')
check('Stable customers exist', 'Stable' in risk_dist['churn_risk_level'].values)

## 3. CLV Model Validation

In [ ]:
print('--- CLV PREDICTOR ---\n')

try:
    clv_eval = q(f'SELECT * FROM ML.EVALUATE(MODEL `{PROJECT}.analytics.clv_predictor`)')
    r2 = float(clv_eval.iloc[0].get('r2_score', 0))
    mae = float(clv_eval.iloc[0].get('mean_absolute_error', 0))
    print(f'  R-squared:           {r2:.4f}')
    print(f'  Mean Absolute Error: R{mae:,.0f}')
    print()

    check('R-squared > 0.30 (model has predictive power)', r2 > 0.30, f'got {r2:.4f}')
    check('R-squared < 0.99 (not overfit)', r2 < 0.99, f'got {r2:.4f}')

except Exception as e:
    print(f'  CLV model not found or failed: {e}')
    warn('CLV model not available - run train_clv_model.sql first')

# CLV predictions sanity
try:
    clv_stats = q(f"""
        SELECT
            COUNT(*) AS total,
            COUNTIF(predicted_clv < 0) AS negative_clv,
            ROUND(MIN(predicted_clv), 0) AS min_clv,
            ROUND(AVG(predicted_clv), 0) AS avg_clv,
            ROUND(APPROX_QUANTILES(predicted_clv, 2)[OFFSET(1)], 0) AS median_clv,
            ROUND(MAX(predicted_clv), 0) AS max_clv
        FROM `{PROJECT}.marts.mart_customer_clv`
    """)
    display(clv_stats)
    s = clv_stats.iloc[0]
    check('No negative CLV predictions', int(s['negative_clv']) == 0, f'{int(s["negative_clv"]):,} negative')
    check('Avg CLV is realistic (R100 - R5M)', 100 < float(s['avg_clv']) < 5e6, f'got R{float(s["avg_clv"]):,.0f}')
    check('Max CLV not absurd (< R50M)', float(s['max_clv']) < 50e6, f'got R{float(s["max_clv"]):,.0f}')

    # tier distribution
    tiers = q(f"""
        SELECT clv_tier, COUNT(*) AS n, ROUND(AVG(predicted_clv), 0) AS avg_clv
        FROM `{PROJECT}.marts.mart_customer_clv`
        GROUP BY clv_tier ORDER BY avg_clv DESC
    """)
    display(tiers)
    check('5 CLV tiers exist', len(tiers) == 5, f'got {len(tiers)}')

except Exception as e:
    warn(f'CLV predictions not available: {e}')

## 4. Data Integrity Checks

In [ ]:
print('--- DATA INTEGRITY ---\n')

# market share sums to ~100% (allow 2% tolerance for rounding)
ms_check = q(f"""
    SELECT CATEGORY_TWO,
           ROUND(SUM(market_share_pct), 1) AS total_market_share
    FROM `{PROJECT}.marts.mart_destination_benchmarks`
    GROUP BY CATEGORY_TWO
    HAVING ABS(SUM(market_share_pct) - 100) > 2
""")
check('Market share sums to ~100% in all categories (2% tolerance)', len(ms_check) == 0,
      f'{len(ms_check)} categories off by >2%')
if not ms_check.empty:
    display(ms_check.head())

# retention never exceeds 100%
ret_check = q(f"""
    SELECT COUNT(*) AS violations
    FROM `{PROJECT}.marts.mart_cohort_retention`
    WHERE retention_pct > 100.1
""")
check('Retention never exceeds 100%', int(ret_check.iloc[0]['violations']) == 0,
      f'{int(ret_check.iloc[0]["violations"]):,} violations')

# no duplicate customers in cluster output
dup_check = q(f"""
    SELECT COUNT(*) - COUNT(DISTINCT UNIQUE_ID) AS duplicates
    FROM `{PROJECT}.marts.mart_cluster_output`
""")
check('No duplicate customers in cluster output', int(dup_check.iloc[0]['duplicates']) == 0,
      f'{int(dup_check.iloc[0]["duplicates"]):,} duplicates')

# all segments have correct names
seg_names = q(f"""
    SELECT DISTINCT segment_name FROM `{PROJECT}.marts.mart_cluster_output`
""")
expected = {'Champions', 'Loyal High Value', 'Steady Mid-Tier', 'At Risk', 'Dormant'}
actual = set(seg_names['segment_name'].tolist())
check('All 5 segment names correct', actual == expected, f'got {actual}')

# no NULL segment names
null_seg = q(f"""
    SELECT COUNTIF(segment_name IS NULL) AS null_count
    FROM `{PROJECT}.marts.mart_cluster_output`
""")
check('No NULL segment names', int(null_seg.iloc[0]['null_count']) == 0)

## 5. Cross-Model Consistency

In [ ]:
print('--- CROSS-MODEL CONSISTENCY ---\n')

# Champions should have lower churn risk than Dormant
try:
    seg_churn = q(f"""
        SELECT co.segment_name,
               ROUND(AVG(cr.churn_probability) * 100, 1) AS avg_churn_pct
        FROM `{PROJECT}.marts.mart_cluster_output` co
        JOIN `{PROJECT}.marts.mart_churn_risk` cr ON co.UNIQUE_ID = cr.UNIQUE_ID
        GROUP BY co.segment_name
        ORDER BY avg_churn_pct
    """)
    display(seg_churn)

    champ_churn = float(seg_churn[seg_churn['segment_name'] == 'Champions']['avg_churn_pct'].values[0])
    dorm_churn = float(seg_churn[seg_churn['segment_name'] == 'Dormant']['avg_churn_pct'].values[0])
    check('Champions have lower churn than Dormant', champ_churn < dorm_churn,
          f'Champions: {champ_churn}%, Dormant: {dorm_churn}%')

except Exception as e:
    warn(f'Could not check segment-churn consistency: {e}')

# CLV should correlate with segments
try:
    seg_clv = q(f"""
        SELECT co.segment_name,
               ROUND(AVG(cv.predicted_clv), 0) AS avg_clv
        FROM `{PROJECT}.marts.mart_cluster_output` co
        JOIN `{PROJECT}.marts.mart_customer_clv` cv ON co.UNIQUE_ID = cv.UNIQUE_ID
        GROUP BY co.segment_name
        ORDER BY avg_clv DESC
    """)
    display(seg_clv)

    champ_clv = float(seg_clv[seg_clv['segment_name'] == 'Champions']['avg_clv'].values[0])
    dorm_clv = float(seg_clv[seg_clv['segment_name'] == 'Dormant']['avg_clv'].values[0])
    check('Champions have higher CLV than Dormant', champ_clv > dorm_clv,
          f'Champions: R{champ_clv:,.0f}, Dormant: R{dorm_clv:,.0f}')

except Exception as e:
    warn(f'Could not check segment-CLV consistency: {e}')

# Spend momentum: declining customers should have higher churn risk
try:
    mom_churn = q(f"""
        SELECT sm.momentum_status,
               ROUND(AVG(cr.churn_probability) * 100, 1) AS avg_churn_pct,
               COUNT(*) AS n
        FROM `{PROJECT}.marts.mart_spend_momentum` sm
        JOIN `{PROJECT}.marts.mart_churn_risk` cr ON sm.UNIQUE_ID = cr.UNIQUE_ID
        GROUP BY sm.momentum_status
        ORDER BY avg_churn_pct DESC
    """)
    display(mom_churn)

    dec_churn = float(mom_churn[mom_churn['momentum_status'] == 'Declining']['avg_churn_pct'].values[0]) if 'Declining' in mom_churn['momentum_status'].values else 0
    acc_churn = float(mom_churn[mom_churn['momentum_status'] == 'Accelerating']['avg_churn_pct'].values[0]) if 'Accelerating' in mom_churn['momentum_status'].values else 100
    check('Declining customers have higher churn than Accelerating', dec_churn > acc_churn,
          f'Declining: {dec_churn}%, Accelerating: {acc_churn}%')

except Exception as e:
    warn(f'Could not check momentum-churn consistency: {e}')

## 6. Summary

In [ ]:
print('\n' + '='*50)
print(f'  QUALITY ASSESSMENT SUMMARY')
print(f'  Passed:   {passed}')
print(f'  Failed:   {failed}')
print(f'  Warnings: {warnings}')
print('='*50)

if failed == 0:
    print('\n  All checks passed. Models and data are consistent.')
elif failed <= 2:
    print('\n  Minor issues detected. Review the flagged items above.')
else:
    print('\n  Several issues detected. Investigate before using these outputs.')